In [1]:
%matplotlib ipympl
from IPython.display import Image, display
import numpy as np
from qutip import *
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from ipywidgets import interact, IntSlider
import matplotlib.ticker as ticker
import numba
from numba import njit, prange
import pickle
import os

## Fidelity
Generic definition : <br>
$ \mathcal{F}\left( \rho, \sigma \right) = \left( Tr \left[ \sqrt{ \sqrt{\rho} \sigma \sqrt{\rho} }\right] \right)^{2} $ <br>
Definition for Pure State : <br>
$ \mathcal{F}\left( \rho, \sigma \right) = |\langle \psi_{\rho} | \psi_{\sigma} \rangle|^{2} $ <br>
Definition for Qubits : <br>
$ \mathcal{F}\left( \rho, \sigma \right) = Tr\left( \rho \, \sigma \right) + 2 \sqrt{Det\left ( \rho \right) \, Det\left ( \sigma \right)} $ <br>
## Trace Distance
Generic definition : <br>
$ \mathcal{T}\left( \rho, \sigma \right) = \frac{1}{2} Tr \left[ \sqrt{\left( \rho - \sigma \right)^{\dagger} \left( \rho - \sigma  \right)} \right] $ <br>
### Relationship
$ 1 - \sqrt{\mathcal{F}\left( \rho, \sigma \right)} \leq \mathcal{T}\left( \rho, \sigma \right) \leq \sqrt{1 - \mathcal{F}\left( \rho, \sigma \right)} $

In [5]:
# ============================================================
# CONFIGURAZIONE GLOBALE
# Cambia MODE per switchare tra i due dataset:
#   'normal'       → results_intermediate.pkl
#   'close_to_90'  → results_close_to_90_deg.pkl
# ============================================================

MODE = 'close_to_90'   # <-- cambia qui: 'normal' oppure 'close_to_90'

# --- Paths ---
input_path = "../Results/Data/"
path_imm   = '/home/francesco/Collisional_Methods/Results/Plot/Probability/Fidelity'

# --- Mapping automatico in base a MODE ---
_cfg = {
    'normal': {
        'input_file': os.path.join(input_path, "results_intermediate_many_trajectories.pkl"),
        'save_fidelity_time_avg': os.path.join(path_imm, "Fidelity_time_avg.png"),
        'save_fidelity_max_value': os.path.join(path_imm, "Fidelity_max_value.png"),
        'save_trace_distance_time_avg': os.path.join(path_imm, "Trace_Distance_time_avg.png"),
        'save_trace_distance_max_value': os.path.join(path_imm, "Trace_Distance_max_value.png"),
        'thetas_deg': [90, 60, 45, 30, 0],
    },
    'close_to_90': {
        'input_file': os.path.join(input_path, "results_close_to_90_deg_many_trajectories.pkl"),
        'save_fidelity_time_avg': os.path.join(path_imm, "Fidelity_time_avg_close_to_90_deg.png"),
        'save_fidelity_max_value': os.path.join(path_imm, "Fidelity_max_value_close_to_90_deg.png"),
        'save_trace_distance_time_avg': os.path.join(path_imm, "Trace_Distance_time_avg_close_to_90_deg.png"),
        'save_trace_distance_max_value': os.path.join(path_imm, "Trace_Distance_max_value_close_to_90_deg.png"),
        'thetas_deg': [90, 89.9, 89.5, 89, 88.5, 88],
    },
}

cfg = _cfg[MODE]

# --- Parametri fisici ---
dt = 0.01
tf = 50.0
time_steps = int(tf / dt)
site_idx = 0   # 0 per Qubit 1, 1 per Qubit 2
N_traj_list = [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]

# --- Caricamento dati ---
with open(cfg['input_file'], 'rb') as f:
    results = pickle.load(f)

print(f"MODE    : {MODE}")
print(f"File    : {cfg['input_file']}")
print(f"Theta   : {[f'{np.degrees(t):.3f}°' for t in results.keys()]}")

MODE    : close_to_90
File    : ../Results/Data/results_close_to_90_deg_many_trajectories.pkl
Theta   : ['90.000°', '89.900°', '89.500°', '89.000°', '88.500°', '88.000°']


In [6]:
# Creiamo il dizionario per contenere i vari tensori 4D
rho_tensors = {}

# Iteriamo su tutti gli angoli (chiavi del dizionario) appena caricati
for theta in results.keys():
    
    # Inizializziamo il mega-array 4D per questo specifico angolo
    # Dimensioni: (len(N_traj_list), time_steps, 2, 2)
    rho_tensor = np.zeros((len(N_traj_list), time_steps, 2, 2), dtype=np.complex128)
    
    for i, N_traj in enumerate(N_traj_list):
        # Per sicurezza, verifichiamo che N_traj esista nei risultati
        if N_traj in results[theta][dt]:
            traj_data = results[theta][dt][N_traj]['trajectory_wf']
            
            avg_pop = traj_data['average_pop']  # Array (2, time_steps)
            avg_coh = traj_data['average_coh']  # Array (2, time_steps)
            
            # --- Costruzione massiva della matrice densità ---
            # Elemento (0,0): Popolazione sito 1
            rho_tensor[i, :, 0, 0] = avg_pop[0, :]
            
            # Elemento (1,1): Popolazione sito 2
            rho_tensor[i, :, 1, 1] = avg_pop[1, :]
            
            # Elemento (0,1): Coerenza
            rho_tensor[i, :, 0, 1] = avg_coh[0, :]
            
            # Elemento (1,0): Coerenza complessa coniugata
            rho_tensor[i, :, 1, 0] = np.conj(avg_coh[0, :])
        
    # Salviamo il tensore nel dizionario
    rho_tensors[theta] = rho_tensor

print("Estrazione delle matrici densità completata!")
print(f"Ho creato {len(rho_tensors)} tensori 4D per la modalità '{MODE}'.")

# Verifica rapida sulla shape del primo elemento
primo_theta = list(results.keys())[0]
print(f"Shape del tensore: {rho_tensors[primo_theta].shape} -> (N_traj, time_steps, 2, 2)")

Estrazione delle matrici densità completata!
Ho creato 6 tensori 4D per la modalità 'close_to_90'.
Shape del tensore: (19, 5000, 2, 2) -> (N_traj, time_steps, 2, 2)
